# Data Comparison Tool

This notebook fetches the results of one SQL Server query and one Redshift query, then creates an Excel reconciliation report. Keep the two queries aligned: they should return the same business columns and the same logical key.

In [ ]:
# Uncomment and run each line once if the package is missing.
# %pip install jupyterlab
# %pip install pandas
# %pip install pyodbc
# %pip install redshift-connector
# %pip install python-dotenv
# %pip install openpyxl

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv
import pandas as pd
import pyodbc
import redshift_connector
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

load_dotenv()  # Loads local credentials from .env. The file is ignored by Git.

# Edit these two queries so that they return matching comparison columns.
SQL_SERVER_QUERY = """
SELECT TOP 1000
    id,
    updated_at,
    status
FROM dbo.your_table;
"""

REDSHIFT_QUERY = """
SELECT
    id,
    updated_at,
    status
FROM your_schema.your_table
LIMIT 1000;
"""

KEY_COLUMNS = ["id"]  # Columns that identify one logical record.
OUTPUT_FILE = Path("outputs/comparison_report.xlsx")

SQL_SERVER_CONNECTION_STRING = os.getenv("SQL_SERVER_CONNECTION_STRING", "")
REDSHIFT_SETTINGS = {
    "host": os.getenv("REDSHIFT_HOST", ""),
    "port": int(os.getenv("REDSHIFT_PORT", "5439")),
    "database": os.getenv("REDSHIFT_DATABASE", ""),
    "user": os.getenv("REDSHIFT_USER", ""),
    "password": os.getenv("REDSHIFT_PASSWORD", ""),
}


In [ ]:
def require_connection_settings():
    if not SQL_SERVER_CONNECTION_STRING:
        raise ValueError("Set SQL_SERVER_CONNECTION_STRING in .env.")

    missing = [name for name, value in REDSHIFT_SETTINGS.items() if not value]
    if missing:
        raise ValueError(f"Set these Redshift values in .env: {', '.join(missing)}")

    if not KEY_COLUMNS:
        raise ValueError("Add at least one column to KEY_COLUMNS.")


def read_sql_server(query):
    connection = pyodbc.connect(SQL_SERVER_CONNECTION_STRING)
    try:
        return pd.read_sql_query(query, connection)
    finally:
        connection.close()


def read_redshift(query):
    connection = redshift_connector.connect(**REDSHIFT_SETTINGS)
    try:
        return pd.read_sql_query(query, connection)
    finally:
        connection.close()


require_connection_settings()
sql_server_data = read_sql_server(SQL_SERVER_QUERY)
redshift_data = read_redshift(REDSHIFT_QUERY)

print(f"Loaded {len(sql_server_data):,} rows from SQL Server.")
print(f"Loaded {len(redshift_data):,} rows from Redshift.")


In [ ]:
def prepare_dataframe(dataframe):
    prepared = dataframe.copy()
    prepared.columns = [str(column).strip().lower() for column in prepared.columns]

    if prepared.columns.duplicated().any():
        duplicates = prepared.columns[prepared.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate column names after normalization: {duplicates}")

    for column in prepared.select_dtypes(include=["object", "string"]).columns:
        values = prepared[column].astype("string").str.strip()
        prepared[column] = values.mask(values == "", pd.NA)
    return prepared


def duplicate_key_rows(dataframe, key_columns):
    return dataframe[dataframe.duplicated(subset=key_columns, keep=False)].sort_values(key_columns)


def column_report(sql_server, redshift):
    columns = sorted(set(sql_server.columns).union(redshift.columns))
    return pd.DataFrame({
        "Column": columns,
        "SQL Server dtype": [str(sql_server[column].dtype) if column in sql_server else "Missing" for column in columns],
        "Redshift dtype": [str(redshift[column].dtype) if column in redshift else "Missing" for column in columns],
        "Present in both": [column in sql_server and column in redshift for column in columns],
    })


def value_differences(sql_server, redshift, key_columns):
    sql_indexed = sql_server.set_index(key_columns, drop=False)
    redshift_indexed = redshift.set_index(key_columns, drop=False)
    common_keys = sql_indexed.index.intersection(redshift_indexed.index, sort=False)
    sql_common = sql_indexed.loc[common_keys]
    redshift_common = redshift_indexed.loc[common_keys]
    shared_columns = sorted(set(sql_server.columns).intersection(redshift.columns).difference(key_columns))

    differences = []
    for column in shared_columns:
        sql_values = sql_common[column]
        redshift_values = redshift_common[column]
        matches = sql_values.eq(redshift_values).fillna(False) | (sql_values.isna() & redshift_values.isna())

        if not matches.all():
            changed = sql_common.loc[~matches, key_columns].copy()
            changed["Column"] = column
            changed["SQL Server value"] = sql_values[~matches].to_numpy()
            changed["Redshift value"] = redshift_values[~matches].to_numpy()
            differences.append(changed)

    if not differences:
        return pd.DataFrame(columns=[*key_columns, "Column", "SQL Server value", "Redshift value"])
    return pd.concat(differences, ignore_index=True)


def compare_sources(sql_server, redshift, key_columns):
    sql_server = prepare_dataframe(sql_server)
    redshift = prepare_dataframe(redshift)
    key_columns = [column.strip().lower() for column in key_columns]

    missing_keys = set(key_columns).difference(sql_server.columns).union(set(key_columns).difference(redshift.columns))
    if missing_keys:
        raise ValueError(f"KEY_COLUMNS are missing from a query result: {', '.join(sorted(missing_keys))}")

    sql_indexed = sql_server.set_index(key_columns, drop=False)
    redshift_indexed = redshift.set_index(key_columns, drop=False)
    sql_only = sql_indexed.loc[sql_indexed.index.difference(redshift_indexed.index)].reset_index(drop=True)
    redshift_only = redshift_indexed.loc[redshift_indexed.index.difference(sql_indexed.index)].reset_index(drop=True)
    sql_duplicate_keys = duplicate_key_rows(sql_server, key_columns)
    redshift_duplicate_keys = duplicate_key_rows(redshift, key_columns)
    differences = value_differences(sql_server, redshift, key_columns)
    columns = column_report(sql_server, redshift)

    passed = (
        columns["Present in both"].all()
        and sql_only.empty
        and redshift_only.empty
        and sql_duplicate_keys.empty
        and redshift_duplicate_keys.empty
        and differences.empty
    )
    summary = pd.DataFrame([
        {"Metric": "Comparison status", "Value": "PASS" if passed else "REVIEW"},
        {"Metric": "SQL Server rows", "Value": len(sql_server)},
        {"Metric": "Redshift rows", "Value": len(redshift)},
        {"Metric": "Columns in both sources", "Value": int(columns["Present in both"].sum())},
        {"Metric": "Rows only in SQL Server", "Value": len(sql_only)},
        {"Metric": "Rows only in Redshift", "Value": len(redshift_only)},
        {"Metric": "Duplicate key rows in SQL Server", "Value": len(sql_duplicate_keys)},
        {"Metric": "Duplicate key rows in Redshift", "Value": len(redshift_duplicate_keys)},
        {"Metric": "Value-level differences", "Value": len(differences)},
    ])

    return {
        "Summary": summary,
        "Column comparison": columns,
        "Only in SQL Server": sql_only,
        "Only in Redshift": redshift_only,
        "Duplicate keys SQL": sql_duplicate_keys,
        "Duplicate keys Redshift": redshift_duplicate_keys,
        "Duplicate rows SQL": sql_server[sql_server.duplicated(keep=False)],
        "Duplicate rows Redshift": redshift[redshift.duplicated(keep=False)],
        "Value differences": differences,
    }


report = compare_sources(sql_server_data, redshift_data, KEY_COLUMNS)
report["Summary"]


In [ ]:
def format_sheet(worksheet, highlight=False):
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions

    header_fill = PatternFill("solid", fgColor="1F4E78")
    warning_fill = PatternFill("solid", fgColor="FCE4D6")
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = Font(color="FFFFFF", bold=True)

    if highlight:
        for row in worksheet.iter_rows(min_row=2):
            for cell in row:
                cell.fill = warning_fill

    for column_cells in worksheet.columns:
        width = min(max(len(str(cell.value or "")) for cell in column_cells) + 2, 45)
        worksheet.column_dimensions[get_column_letter(column_cells[0].column)].width = width


OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for sheet_name, dataframe in report.items():
        dataframe.to_excel(writer, sheet_name=sheet_name, index=False)
        format_sheet(writer.sheets[sheet_name], highlight=sheet_name == "Value differences")

print(f"Saved comparison report: {OUTPUT_FILE}")
